In [1]:
%pip install pdfplumber openpyxl

692.11s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


     |████████████████████████████████| 60 kB 1.2 MB/s eta 0:00:011
     |████████████████████████████████| 250 kB 538 kB/s eta 0:00:01
     |████████████████████████████████| 3.5 MB 97 kB/s eta 0:00:014
     |████████████████████████████████| 60 kB 66 kB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 81 kB/s eta 0:00:016
     |████████████████████████████████| 60 kB 117 kB/s ta 0:00:011
     |████████████████████████████████| 5.6 MB 380 kB/s eta 0:00:01
     |████████████████████████████████| 3.9 MB 1.8 MB/s eta 0:00:01
     |████████████████████████████████| 374 kB 2.4 MB/s eta 0:00:01
     |████████████████████████████████| 180 kB 2.9 MB/s eta 0:00:01
     |████████████████████████████████| 118 kB 3.3 MB/s eta 0:00:01
You should consider upgrading via the '/Users/myankbeniwal/Documents/ChatGPT/STT AI/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pdfplumber
from openpyxl import load_workbook

folder = Path("files")

# Extract PDF text and tables by page.
pdf_pages = []

for path in sorted(folder.glob("*.pdf")):
    with pdfplumber.open(path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            pdf_pages.append({
                "file": path.name,
                "page": page_number,
                "text": page.extract_text() or "",
                "tables": page.extract_tables(),
            })

# Load all Excel sheets, retaining access to styles and merged cells.
excel_path = next(folder.glob("*.xlsx"))
workbook = load_workbook(excel_path, data_only=False)

for sheet in workbook:
    print(sheet.title, "merged ranges:", list(sheet.merged_cells.ranges))

Time table merged ranges: [<MergedCellRange A124:R124>, <MergedCellRange A256:R256>, <MergedCellRange A187:R187>, <MergedCellRange A87:R87>, <MergedCellRange A172:R172>, <MergedCellRange A183:R183>, <MergedCellRange A133:R133>, <MergedCellRange A108:R108>, <MergedCellRange A286:R286>, <MergedCellRange A118:R118>, <MergedCellRange A164:R164>, <MergedCellRange A365:R365>, <MergedCellRange A385:R385>, <MergedCellRange A301:R301>, <MergedCellRange A179:R179>, <MergedCellRange A239:R239>, <MergedCellRange A2:R2>, <MergedCellRange A71:R71>, <MergedCellRange A204:R204>, <MergedCellRange A23:R23>, <MergedCellRange A17:R17>, <MergedCellRange A262:R262>, <MergedCellRange A140:R140>, <MergedCellRange A93:R93>, <MergedCellRange A102:R102>, <MergedCellRange A348:R348>, <MergedCellRange A329:R329>, <MergedCellRange A157:R157>, <MergedCellRange A77:R77>, <MergedCellRange A310:R310>, <MergedCellRange A148:R148>, <MergedCellRange A191:R191>, <MergedCellRange A225:R225>, <MergedCellRange A61:R61>, <Merg

### Inspect extracted pages before cleaning

Use `show_pdf_pages` to review text and tables. Each preview shows the original
`pdf_pages` list index so you can locate the record later. The function does not
change the extracted data. Filters select pages; `max_chars` only shortens the preview.

Tables may also appear in the extracted page text. Review this duplication before
chunking. These previews show extracted content, not the original PDF colours or layout.


In [3]:
def show_pdf_pages(pdf_pages, *, file=None, page=None, start=0, limit=1,
                   max_chars=None, show_tables=True):
    """Preview extracted pages without modifying them.

    file: case-insensitive filename substring.
    page: original PDF page number (1-based), optionally combined with file.
    start: zero-based offset within matching records.
    limit: number of matching records to display; None displays all.
    max_chars: text preview length; None shows the full text.
    show_tables: display extracted tables separately from page text.
    """
    from html import escape
    from IPython.display import HTML, display

    if start < 0 or (limit is not None and limit < 1):
        raise ValueError("start must be >= 0 and limit must be >= 1 or None")
    if max_chars is not None and max_chars < 1:
        raise ValueError("max_chars must be >= 1 or None")

    matches = [
        (index, record) for index, record in enumerate(pdf_pages)
        if (file is None or file.casefold() in record["file"].casefold())
        and (page is None or record["page"] == page)
    ]
    selected = matches[start:] if limit is None else matches[start:start + limit]
    print(f"Showing {len(selected)} of {len(matches)} matching pages "
          f"({len(pdf_pages)} total).")
    if not selected:
        print("No pages to display. Check the filters or start offset.")
        return

    for index, record in selected:
        text = record.get("text") or ""
        tables = record.get("tables") or []
        print("\n" + "=" * 80)
        print(f"pdf_pages[{index}] | {record['file']} | Page {record['page']}")
        print(f"Characters: {len(text):,} | Tables: {len(tables)}")
        print("-" * 80)
        print(text[:max_chars] if text else "[No extracted text]")
        if max_chars is not None and len(text) > max_chars:
            print(f"\n[Preview truncated: {len(text) - max_chars:,} characters "
                  "hidden. Use max_chars=None for full text.]")

        if show_tables:
            for table_number, table in enumerate(tables, start=1):
                print(f"\nTable {table_number} ({len(table)} rows):")
                # Keep every row as data; do not assume the first row is a header.
                rows = []
                for row in table:
                    cells = "".join(
                        '<td style="border:1px solid #aaa;padding:6px;'
                        'white-space:pre-wrap;text-align:left;vertical-align:top">'
                        + escape("" if value is None else str(value)) + "</td>"
                        for value in row
                    )
                    rows.append("<tr>" + cells + "</tr>")
                display(HTML('<div style="overflow-x:auto"><table '
                             'style="border-collapse:collapse">'
                             + "".join(rows) + "</table></div>"))


In [5]:
# Preview the first page with full text and tables.
show_pdf_pages(pdf_pages, limit=None) 

# More examples (uncomment one at a time):
# show_pdf_pages(pdf_pages, start=1, limit=2)
# show_pdf_pages(pdf_pages, file="Revised CSE", page=1)
# show_pdf_pages(pdf_pages, file="Adv5_", limit=None)
# show_pdf_pages(pdf_pages, max_chars=1000, show_tables=False)


Showing 16 of 16 matching pages (16 total).

pdf_pages[0] | Adv17_Courses and Credit System.pdf | Page 1
Characters: 2,112 | Tables: 0
--------------------------------------------------------------------------------
INDIAN INSTITUTE OF TECHNOLOGY GANDHINAGAR
COURSES AND CREDIT SYSTEM
Course Number and Title
Each course has a unique number consisting of a code to identify the discipline with 2 or 3
characters of English alphabet followed by a three-digit number.
HS:000 Humanities and Social Science
ES:000 Engineering Sciences
IN:000 Institute Electives
LS:000 Biological Engineering
CL:000 Chemical Engineering
CE:000 Civil Engineering
EE:000 Electrical Engineering
MSE:000 Materials Science and Engineering
ME:000 Mechanical Engineering
CS:000 Computer Sciences and Engineering
MS:000 Management Studies
CH:000 Chemistry
PH:000 Physics
MA:000 Mathematics
The first digit following the alphabetic code represents the level of the course.
XX:100 First year B.Tech level course
XX:200 Second year 

Type of Courses Basis of Credits,Contact Hours Distribution/Week,,,Approximate Contact Hours in a Semester,Credits
,Lecture,Tutorial,Lab,,
Regular theory and Laboratory Courses 10 Lectures for 1 credit No credit for tutorial 2-3 Lab hours per week for 1 credit,2 3 3 4 0 0 2 3 3,1 0 1 0 0 0 0 0 1,0 0 0 0 2-3 4 3 3 2,28 + 12 + 0 40 + 0 + 0 40 + 12 + 0 52 + 0 + 0 0 + 0 + 20-30 0 + 0 + 40 28 + 0 + 22 40 + 0 + 20 40 + 12 + 20,3 4 4 5 1 2 4 5 5
Short Module:,8-10 Lectures 16-20 Lectures,,,,1 2



pdf_pages[2] | Adv24_Norms For Pass_Fail_Option_And_Substitution_Of_Courses.pdf | Page 1
Characters: 2,842 | Tables: 0
--------------------------------------------------------------------------------
INDIAN INSTITUTE OF TECHNOLOGY GANDHINAGAR
NORMS FOR PASS/FAIL OPTION AND SUBSTITUTION OF COURSES
(As approved by Senate in its 15th meeting held on 03 January 2014 and
in its 66th meeting held on 16 October 2023)
It is felt important to encourage students to explore courses in areas in which they have little
background and promote additional learning among the students through courses beyond
minimum graduation requirements. It will be useful to have provisions against decline in overall
performance indices of the students. In view of this, the Senate approved the following
provisions in the curriculum of all programmes.
A. PASS/FAIL OPTION IN COURSES
1. A student can submit request to convert his/her earlier registered course(s) with
passing grade(s) to Pass/Fail option. Each student wil

,,,,,,,,
,,,,MTech (Thesis Track),,,MTech (Coursework Track),
,,,,,,,,
,,,>=32,,,>=48,,
,Credits through,,,,,,,
,courses,,,,,,,
,,,,,,,,
,,,"The coursework will include 1. 4 credits of Writing course, 2. >= 4 credits of HSS courses [Applicable for students admitted before AY 2025-26], 3. 4 credits of Applications of AI courses relevant to the discipline [Applicable for students admitted from AY 2025-26], 4. >= 8 credits of electives, 5. >= 16 credits of courses specified by the discipline, 6. 2 physical education courses of zero credit each.",,,,,
,,,,,,,,
,,,,,,,,
,,,>=32,,,0,,



Table 2 (2 rows):


...


Table 3 (2 rows):


Further
expectations



pdf_pages[8] | Adv5_NormsforMTechProgramme.pdf | Page 5
Characters: 2,276 | Tables: 0
--------------------------------------------------------------------------------
▪ In case the CPI of a student falls below 6.0 out of 10, the fellowship provided, if
any, by the Institute will be stopped immediately and for the future.
▪ If a student accumulates F grades in more than 20 credits of thesis units any time
during their MTech programme, the student will have to exit the programme.
(Applicable for Students Admitted before AY 2023-24)
▪ The minimum CPI for graduation is 6.0 out of 10.
▪ The minimum CPI for continuing in the programme at any point is 5.5 out of 10.
That is, if at any stage, the student has a CPI less than 5.5, the programme will be
terminated.
▪ In case the CPI of a student falls below 6.0 or the student accumulates three or more
Us towards thesis grades at any point of time, the fellowship provided by the Institute
will be stopped permanently. The student may seek financia

AI/ML,Systems,Theory
Core: ES 678: Machine Learning ES 667: Deep Learning,Core: CS 612: Computer Systems CS 616: Distributed Systems and Cloud Computing,Core: CS 619: CS Theory Toolkit CS 614: Advanced Algrithms
AI/ML Basket,Systems Basket,Theory Basket



pdf_pages[15] | Revised CSE MTech Course Structure [2025-26].pdf | Page 2
Characters: 1,065 | Tables: 0
--------------------------------------------------------------------------------
13. ES 634: Applied Multivariate Data Analysis
14. ES 615: Nature-inspired computing,
Systems Basket
1. CS 615: Advanced Networks
2. ES 417: 5G and Beyond
3. CS 434: Software Engineering and Testing
4. CS 435: Human-Computer Interaction
5. CS 431: Computer and Network Security
6. CS 691-XIV: Special Topics: Usable Privacy and Security
Theory Basket
1. CS 617: Complexity Theory
2. CS 607: Combinatorics in CS
3. ES 627: Linear Algebra and Computation
4. ES 643: Quantum computation and information
5. MA 607: Graph Theory
Additionally, according to the new MTech credit structure, the remaining credits for the degree
requirement can be fulfilled by any of the above courses or the following options.
1. Any CS xxx tagged course (offline)
2. Any of the above courses that are mentioned in the above baskets.
For 